In [1]:
import json
import random

# ============================================================
# 1. CONFIGURATION
# ============================================================
INPUT_FILE = "experiment_dataset.jsonl"
OUTPUT_FILE = "balanced_experiment_dataset.jsonl"
TARGET_PER_DIFFICULTY = 50

# Storage buckets for sorting the dataset rows
buckets = {
    "easy": [],
    "medium": [],
    "hard": []
}

# ============================================================
# 2. READ AND SORT COHORTS
# ============================================================
print(f"Scanning and sorting records from '{INPUT_FILE}'...")

with open(INPUT_FILE, 'r', encoding='utf-8') as infile:
    for line_num, line in enumerate(infile, 1):
        line = line.strip()
        if not line:
            continue
            
        try:
            record = json.loads(line)
            difficulty = str(record.get("difficulty", "")).lower().strip()
            
            if difficulty in buckets:
                buckets[difficulty].append(record)
            else:
                # Catch any unexpected difficulty tags if they exist
                print(f"⚠️ Line {line_num}: Unknown difficulty tier '{difficulty}'. Skipping tag...")
                
        except json.JSONDecodeError:
            print(f"⚠️ Line {line_num}: Invalid JSON row. Skipping syntax error...")

# ============================================================
# 3. VERIFY DATASET VIABILITY
# ============================================================
print("\n--- DATASET COMPOSITION ---")
can_proceed = True

for level, items in buckets.items():
    print(f"• {level.capitalize()} items found: {len(items)}")
    if len(items) < TARGET_PER_DIFFICULTY:
        print(f"  ❌ Critical: Need at least {TARGET_PER_DIFFICULTY} items, but only found {len(items)}.")
        can_proceed = False

if not can_proceed:
    print("\n🛑 Execution halted: The dataset does not have enough rows to fulfill a balanced split.")
    exit(1)

# ============================================================
# 4. STRATIFIED SAMPLING & EXPORT
# ============================================================
print(f"\nExtracting exactly {TARGET_PER_DIFFICULTY} random items from each tier...")
balanced_pool = []

# Perform the deterministic random pull for each group
for level, items in buckets.items():
    sampled_tier = random.sample(items, TARGET_PER_DIFFICULTY)
    balanced_pool.extend(sampled_tier)

# Shuffle the combined lists so you don't evaluate all 50 easies consecutively
random.shuffle(balanced_pool)

print(f"Writing balanced subset directly to '{OUTPUT_FILE}'...")
with open(OUTPUT_FILE, 'w', encoding='utf-8') as outfile:
    for record in balanced_pool:
        outfile.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"\n🎉 Success! Crafted a stratified dataset containing exactly {len(balanced_pool)} rows.")

Scanning and sorting records from 'experiment_dataset.jsonl'...

--- DATASET COMPOSITION ---
• Easy items found: 173
• Medium items found: 173
• Hard items found: 173

Extracting exactly 50 random items from each tier...
Writing balanced subset directly to 'balanced_experiment_dataset.jsonl'...

🎉 Success! Crafted a stratified dataset containing exactly 150 rows.


In [2]:
pip install pandas openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]1/2 [openpyxl]
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

def jsonl_to_xlsx(input_file_path, output_file_path):
    # Read the JSONL file
    # lines=True tells pandas to read each line as a separate JSON object
    df = pd.read_json(input_file_path, lines=True)
    
    # Write to Excel
    # index=False prevents pandas from writing row numbers into the file
    df.to_excel(output_file_path, index=False)
    
# Usage
jsonl_to_xlsx('balanced_experiment_dataset.jsonl', 'balanced_experiment_dataset.xlsx')